# 01 · DeepFashion2 — EDA, cleaning & working subset

Refactored from the original `df2_kaggle.py` notebook. It runs on **Kaggle** (recommended) or Colab against the exact dataset
[`thusharanair/deepfashion2-original-with-dataframes`](https://www.kaggle.com/datasets/thusharanair/deepfashion2-original-with-dataframes).

The cleaning, sampling and parsing logic now lives in the tested `fvs_ml` package, so this notebook and the indexing pipeline
(`02_build_index_kaggle.ipynb`) produce **the same** catalogue.

### Fixes to the original notebook
| Original behaviour | Problem | Fix |
|---|---|---|
| `resolve_image_path` matched files by **basename** | `train/image/000001.jpg` and `validation/image/000001.jpg` are different photos → wrong image attached | rows keep `split` + a path relative to the dataset root |
| `find_column` used **substring** matches | candidate `"id"` matched `pair_id` / `category_id` → `item_id` mis-detected | exact, schema-based names (`fvs_ml.annotations.map_csv_columns`) |
| `df.drop_duplicates()` on all columns | fails / misbehaves on list-valued columns (bbox, landmarks) | dedupe on `item_id` |
| balanced sampling `TARGET // 13` per class | rare classes (e.g. short sleeve outwear) can't fill their share → subset falls short of 50k | water-filling quotas redistribute the remainder |
| no held-out split | PRD §7.2 needs an evaluation slice never indexed or tuned on | pair-grouped hold-out from the validation split |

In [ ]:
# Kaggle: Add the dataset via "Add Input", enable Internet. Colab: kagglehub downloads it.
import os, sys, subprocess
REPO_URL = "https://github.com/<your-username>/fashion-visual-search.git"   # ← set after you push the repo
if not os.path.exists("fashion-visual-search"):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "fashion-visual-search/ml", "--no-deps"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "faiss-cpu", "open_clip_torch", "pyarrow"], check=True)

In [ ]:
from pathlib import Path
KAGGLE_INPUT = Path("/kaggle/input/deepfashion2-original-with-dataframes")
if KAGGLE_INPUT.exists():
    DATASET_PATH = KAGGLE_INPUT
else:  # Colab / local
    import kagglehub
    DATASET_PATH = Path(kagglehub.dataset_download("thusharanair/deepfashion2-original-with-dataframes"))
print(DATASET_PATH)

## 1 · What came with the dataset
List the dataframe files shipped with this Kaggle mirror and how their columns map onto the DeepFashion2 schema (exact matching only).

In [ ]:
import pandas as pd
from fvs_ml.annotations import discover_splits, map_csv_columns

frames = sorted(p for p in DATASET_PATH.rglob("*") if p.suffix.lower() in {".csv", ".parquet"})
for p in frames:
    cols = list(pd.read_csv(p, nrows=3).columns) if p.suffix == ".csv" else list(pd.read_parquet(p).columns)
    print(p.relative_to(DATASET_PATH), "→", map_csv_columns(cols))

layout = discover_splits(DATASET_PATH)
for split, (img, ann) in layout.items():
    print(f"{split:10s} images: {img.relative_to(DATASET_PATH)}  annos: {ann.relative_to(DATASET_PATH)}")

## 2 · Load item-level annotations
Every `itemN` in every per-image JSON becomes one row. This is the authoritative source (category, bbox, style, pair_id, source, scale, occlusion, viewpoint).

In [ ]:
from fvs_ml.annotations import load_items
items = load_items(DATASET_PATH, splits=("train", "validation"))
print(f"{len(items):,} items from {items[['split','image_id']].drop_duplicates().shape[0]:,} images")
items.head()

## 3 · EDA

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
items["category"].value_counts().sort_values().plot.barh(ax=axes[0], color="#9f1239")
axes[0].set_title("Items per category"); axes[0].set_xlabel("items")
items.groupby(["split", "source"]).size().unstack().plot.bar(ax=axes[1], rot=0, color=["#9f1239", "#2a78d6"])
axes[1].set_title("Shop vs consumer photos per split")
plt.tight_layout(); plt.show()
(items["category"].value_counts(normalize=True) * 100).round(2).to_frame("% of items")

In [ ]:
from fvs_ml.config import OCCLUSION, SCALE, VIEWPOINT
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (col, names) in zip(axes, [("scale", SCALE), ("occlusion", OCCLUSION), ("viewpoint", VIEWPOINT)]):
    items[col].map(names).value_counts().plot.bar(ax=ax, rot=0, color="#9f1239"); ax.set_title(col.capitalize())
plt.tight_layout(); plt.show()

In [ ]:
items["bbox_area"] = (items.bbox_x2 - items.bbox_x1) * (items.bbox_y2 - items.bbox_y1)
items["bbox_aspect"] = (items.bbox_x2 - items.bbox_x1) / (items.bbox_y2 - items.bbox_y1)
items[["bbox_area", "bbox_aspect"]].describe().T

## 4 · Conservative cleaning
Removed: duplicates, missing essentials, unknown categories, degenerate or tiny boxes (<32 px). **Kept:** consumer photos, side/back views, occlusion and rare classes, because real queries look like that.

In [ ]:
from fvs_ml.clean import clean_items
clean, report = clean_items(items, min_box_side=32)
pd.Series(report).to_frame("rows")

## 5 · Held-out evaluation slice + balanced working subset

In [ ]:
from fvs_ml.subset import holdout_eval_slice, sample_working_subset
TARGET_ITEMS = 50_000   # PRD §7.2 — start with 5_000 while iterating
pool, eval_slice = holdout_eval_slice(clean, fraction=0.25, max_images=4_000, seed=42)
subset = sample_working_subset(pool, TARGET_ITEMS, seed=42)
print(f"subset: {len(subset):,} items · eval slice: {len(eval_slice):,} items "
      f"({eval_slice.pair_id.nunique():,} pairs) · overlap: {len(set(subset.item_id) & set(eval_slice.item_id))}")
subset["category"].value_counts().sort_values().plot.barh(figsize=(8, 5), color="#9f1239", title="Working subset by category");

## 6 · Visual sanity check: bounding-box crops

In [ ]:
from fvs_ml.crop import crop_garment
sample = subset.sample(12, random_state=0)
fig, axes = plt.subplots(3, 4, figsize=(12, 10))
for ax, row in zip(axes.flat, sample.itertuples()):
    ax.imshow(crop_garment(DATASET_PATH / row.image_relpath, (row.bbox_x1, row.bbox_y1, row.bbox_x2, row.bbox_y2), 0.06))
    ax.set_title(row.category, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
from pathlib import Path
OUT = Path("/kaggle/working/processed") if Path("/kaggle/working").exists() else Path("processed")
OUT.mkdir(parents=True, exist_ok=True)
clean.to_parquet(OUT / "deepfashion2_clean_items.parquet")
subset.to_parquet(OUT / "deepfashion2_working_subset.parquet")
eval_slice.to_parquet(OUT / "deepfashion2_eval_slice.parquet")
print("saved to", OUT)